# Corpus Ingestor — mod/onfalo · REGULACION
**Tenant:** `scriptorium` · **Database:** `mod-onfalo`

Ingesta los 13 documentos del proyecto REGULACION en 5 colecciones Chroma.

| Colección | Documentos |
|-----------|------------|
| `mo_regulacion_discursos` | WGS Dubai — intervención Sánchez, feb 2026 |
| `mo_regulacion_articulos` | 07 · 09 · 09a — artículos y análisis |
| `mo_regulacion_fichas` | 09b fichas enciclopédicas · 13 one-pager |
| `mo_regulacion_propuestas` | 12 policy brief · 10 currícula · España Crece (×2) |
| `mo_regulacion_cartas` | 10 carta abierta · 11 tres cartas 2050 · 08 utopía |

**Requisitos:** `pip install chromadb`

Ejecutar celdas en orden. La ingesta es idempotente: reejecutar no duplica chunks.

In [1]:
%pip install -q chromadb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import chromadb
import re
from pathlib import Path

HOME = Path.home()
OASIS = HOME / 'OASIS' / 'aleph-scriptorium'
STORAGE_PATH = str(OASIS / 'ARCHIVO' / 'PLUGINS' / 'VECTOR_MACHINE' / 'STORAGE')
REGULACION_PATH = OASIS / 'onfalo-asesor-sdk' / 'PROYECTOS' / 'REGULACION'

client = chromadb.PersistentClient(path=STORAGE_PATH)
print('Colecciones existentes:', [c.name for c in client.list_collections()])
print('REGULACION_PATH exists:', REGULACION_PATH.exists())

Colecciones existentes: ['mr_analisis', 'me_mapas', 'me_mapas_mapas_sing', 'mr_corpus', 'ml_hilo_narrativo', 'me_turin', 'me_psicoanalisis', 'me_escano', 'ml_piezas_media', 'me_decoherencia', 'ml_recursos', 'mr_editoriales', 'ml_eventos', 'ml_personajes', 'mr_guiones', 'mr_poemas']
REGULACION_PATH exists: True


## Funciones de chunking e ingesta

In [3]:
NL = chr(10)
NL2 = chr(10) + chr(10)


def chunk_markdown(text, max_chars=700):
    """Divide un markdown por secciones ##, luego por párrafo si supera max_chars."""
    parts = re.split(r'(?m)^(#{1,3} .+)', text)
    chunks = []
    title = ''
    body = ''
    for part in parts:
        if re.match(r'^#{1,3} ', part):
            if body.strip():
                chunks.append({'title': title, 'text': body.strip()})
            title = part.strip()
            body = ''
        else:
            body += part
    if body.strip():
        chunks.append({'title': title, 'text': body.strip()})

    final = []
    for chunk in chunks:
        if len(chunk['text']) <= max_chars:
            final.append(chunk)
        else:
            paras = [p.strip() for p in chunk['text'].split(NL2) if p.strip()]
            buf = ''
            for para in paras:
                if len(buf) + len(para) > max_chars and buf:
                    final.append({'title': chunk['title'], 'text': buf.strip()})
                    buf = para
                else:
                    buf += (NL2 if buf else '') + para
            if buf:
                final.append({'title': chunk['title'], 'text': buf.strip()})

    return [c for c in final if len(c['text']) >= 50]


def ingest_file(col_name, filepath, bloque, tipo, id_prefix):
    """Lee un .md, lo chunkea e ingesta en col_name. Idempotente por id."""
    col = client.get_or_create_collection(col_name)
    existing = set(col.get()['ids'])

    text = Path(filepath).read_text(encoding='utf-8')
    chunks = chunk_markdown(text)

    ids, docs, metas = [], [], []
    for i, chunk in enumerate(chunks):
        doc_id = f'{id_prefix}-{i+1:02d}'
        if doc_id in existing:
            continue
        label = (chunk['title'][:40] if chunk['title'] else f'{id_prefix}-{i+1:02d}')
        ids.append(doc_id)
        docs.append(chunk['text'])
        metas.append({'bloque': bloque, 'tipo': tipo, 'marca': label})

    if ids:
        col.add(ids=ids, documents=docs, metadatas=metas)
    print(f'  {col_name:35s} / {id_prefix}: +{len(ids)} chunks  (total {col.count()})')
    return len(ids)

## Colección 1 — `mo_regulacion_discursos`

Intervención de Pedro Sánchez en el WGS Dubai, 3 feb 2026. Texto largo: se chunkea en secciones de ~700 chars.

In [4]:
COL = 'mo_regulacion_discursos'
ingest_file(COL,
            REGULACION_PATH / '260203_intervencion_WGS_dubai.md',
            bloque='WGS-Dubai-2026', tipo='discurso', id_prefix='rdi-wgs')

  mo_regulacion_discursos             / rdi-wgs: +18 chunks  (total 18)


18

## Colección 2 — `mo_regulacion_articulos`

Artículos y análisis: doc 07 (redes sociales + piernas del personaje), 09 y 09a (regulación redes sociales I y II).

In [5]:
COL = 'mo_regulacion_articulos'
ingest_file(COL,
            REGULACION_PATH / '07_Regulacion_redes_sociales_y_piernas_del_personaje.md',
            bloque='doc-07', tipo='analisis', id_prefix='rat-07')
ingest_file(COL,
            REGULACION_PATH / '09_articulo_regulacion_redes_sociales.md',
            bloque='doc-09', tipo='articulo', id_prefix='rat-09')
ingest_file(COL,
            REGULACION_PATH / '09a_articulo_regulacion_redes_sociales_II.md',
            bloque='doc-09a', tipo='articulo', id_prefix='rat-09a')

  mo_regulacion_articulos             / rat-07: +45 chunks  (total 45)
  mo_regulacion_articulos             / rat-09: +51 chunks  (total 96)
  mo_regulacion_articulos             / rat-09a: +55 chunks  (total 151)


55

## Colección 3 — `mo_regulacion_fichas`

Fichas enciclopédicas (09b) + one-pager del proyecto (13). Fragmentos más cortos y estructurados.

In [6]:
COL = 'mo_regulacion_fichas'
ingest_file(COL,
            REGULACION_PATH / '09b_fichas_enciclopedicas_regulacion_redes.md',
            bloque='doc-09b', tipo='ficha', id_prefix='rfi-09b')
ingest_file(COL,
            REGULACION_PATH / '13_one_pager_proyecto.md',
            bloque='doc-13', tipo='one-pager', id_prefix='rfi-13')

  mo_regulacion_fichas                / rfi-09b: +62 chunks  (total 62)
  mo_regulacion_fichas                / rfi-13: +7 chunks  (total 69)


7

## Colección 4 — `mo_regulacion_propuestas`

Policy brief (12), currícula mundial (10), informes España Crece (informe + recalibración).

In [7]:
COL = 'mo_regulacion_propuestas'
ingest_file(COL,
            REGULACION_PATH / '12_policy_brief_gobernanza_algoritmica.md',
            bloque='doc-12', tipo='policy-brief', id_prefix='rpr-12')
ingest_file(COL,
            REGULACION_PATH / '10_curricula_alfabetizacion_digital_mundial.md',
            bloque='doc-10c', tipo='curricula', id_prefix='rpr-10c')
ingest_file(COL,
            REGULACION_PATH / 'ESPANA_CRECE_INFORME.md',
            bloque='espana-crece', tipo='informe', id_prefix='rpr-eci')
ingest_file(COL,
            REGULACION_PATH / 'ESPANA_CRECE_RECALIBRACION.md',
            bloque='espana-crece', tipo='recalibracion', id_prefix='rpr-ecr')

  mo_regulacion_propuestas            / rpr-12: +31 chunks  (total 31)
  mo_regulacion_propuestas            / rpr-10c: +134 chunks  (total 165)
  mo_regulacion_propuestas            / rpr-eci: +39 chunks  (total 204)
  mo_regulacion_propuestas            / rpr-ecr: +66 chunks  (total 270)


66

## Colección 5 — `mo_regulacion_cartas`

Carta abierta calibrada (10), tres cartas desde 2050 (11, ficción archivística), utopía cinco décadas (08).

In [8]:
COL = 'mo_regulacion_cartas'
ingest_file(COL,
            REGULACION_PATH / '10_carta_abierta_calibrada.md',
            bloque='doc-10k', tipo='carta', id_prefix='rca-10k')
ingest_file(COL,
            REGULACION_PATH / '11_tres_cartas_desde_2050.md',
            bloque='doc-11', tipo='carta-ficcion', id_prefix='rca-11')
ingest_file(COL,
            REGULACION_PATH / '08_Utopia_cinco_decadas.md',
            bloque='doc-08', tipo='utopia', id_prefix='rca-08')

  mo_regulacion_cartas                / rca-10k: +69 chunks  (total 69)
  mo_regulacion_cartas                / rca-11: +28 chunks  (total 97)
  mo_regulacion_cartas                / rca-08: +42 chunks  (total 139)


42

## Verificación final

Lista todas las colecciones `mo_regulacion_*` con recuento de chunks.

In [9]:
print('=== mod-onfalo / REGULACION ===')
total = 0
for c in sorted(client.list_collections(), key=lambda x: x.name):
    if c.name.startswith('mo_regulacion'):
        col = client.get_collection(c.name)
        n = col.count()
        total += n
        print(f'  {c.name:35s}  {n:4d} chunks')
print(f'  {"TOTAL":35s}  {total:4d} chunks')

=== mod-onfalo / REGULACION ===
  mo_regulacion_articulos               151 chunks
  mo_regulacion_cartas                  139 chunks
  mo_regulacion_discursos                18 chunks
  mo_regulacion_fichas                   69 chunks
  mo_regulacion_propuestas              270 chunks
  TOTAL                                 647 chunks


In [11]:
# ── Purga colecciones me_* (basurilla de sesiones anteriores) ────────────────
to_delete = [c.name for c in client.list_collections() if c.name.startswith('me_')]
print('Colecciones a eliminar:', to_delete)
for name in to_delete:
    client.delete_collection(name)
    print(f'  ✗ {name} eliminada')
remaining = [c.name for c in client.list_collections()]
print('\\nColecciones restantes:', remaining)

Colecciones a eliminar: []
\nColecciones restantes: ['mo_regulacion_propuestas', 'mr_analisis', 'mo_regulacion_discursos', 'mr_corpus', 'ml_hilo_narrativo', 'mo_regulacion_cartas', 'ml_piezas_media', 'mo_regulacion_fichas', 'mo_regulacion_articulos', 'ml_recursos', 'mr_editoriales', 'ml_eventos', 'ml_personajes', 'mr_guiones', 'mr_poemas']
